In [52]:
import json
import pandas as pd
import xarray as xr
import numpy as np
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import geopandas as gpd
import dask

In [53]:
records = []
list_years = list(range(2012, 2025))
for year in list_years:
    with open(f"./data/GFW_SWA_AIS_{year}.json") as f:
        raw = json.load(f)

    # Extract the list of vessel records
    entries = raw["entries"][0]["public-global-fishing-effort:v3.0"]
    records.extend(entries)  # Append to the main records list

    df1 = pd.DataFrame(records)

In [54]:
df1.head()

,callsign,dataset,date,entryTimestamp,exitTimestamp,firstTransmissionDate,flag,geartype,hours,imo,lastTransmissionDate,lat,lon,mmsi,shipName,vesselId,vesselType
0,LW2678,public-global-vessel-identity:v3.0,2012-06,2012-04-01T05:00:00Z,2012-10-10T12:00:00Z,2012-01-22T02:28:56Z,ARG,TRAWLERS,7.564167,,2016-05-21T04:11:02Z,-46.5,-61.599998,701006445,API V,efb01f7c3-3c18-8305-3408-e8856af57ec0,FISHING
1,LW9298,public-global-vessel-identity:v3.0,2012-04,2012-01-31T00:00:00Z,2012-05-25T02:00:00Z,2012-01-24T12:34:23Z,ARG,SQUID_JIGGER,1.605833,,2016-10-16T02:02:39Z,-53.0,-65.900002,701000907,CORAL BLANCO,031a5d6d0-083e-9fc3-12aa-174cbb8c6894,FISHING
2,LW 2544,public-global-vessel-identity:v3.0,2012-06,2012-03-18T04:00:00Z,2012-12-07T17:00:00Z,2012-01-01T04:35:51Z,ARG,TRAWLERS,6.427778,,2016-05-10T13:46:11Z,-45.2,-64.599998,701094000,ARGENOVA 22,9e198e0c5-5bb9-4b58-9295-9328c00f172c,FISHING
3,LW 8821,public-global-vessel-identity:v3.0,2012-03,2012-03-04T15:00:00Z,2012-12-30T19:00:00Z,2012-01-04T08:56:11Z,ARG,TRAWLERS,2.076667,,2014-12-05T19:58:46Z,-46.0,-61.700001,701000885,POLARBORG II,1d0e25137-755d-7acf-55c1-b1618e5ac155,FISHING
4,LW 9499,public-global-vessel-identity:v3.0,2012-09,2012-04-03T04:00:00Z,2012-10-18T09:00:00Z,2012-01-01T06:14:51Z,ARG,TRAWLERS,1.553333,,2018-06-06T17:01:37Z,-45.3,-64.500000,701000742,UCHI,ac677749e-e0e1-efa9-bc4c-6c9cce9530ad,FISHING


In [55]:

df1["date"] = pd.to_datetime(df1["date"]) # Convert the str time to datetime
df1 = df1[["date", "lat", "lon", "hours", "flag", "geartype"]] # Select only the necessary columns

df1 = df1.replace(r"^\s*$", np.nan, regex=True) # Replace empty strings with NaN
df1 = df1.dropna(subset=["lat", "lon", "date", "flag", "geartype"]) # Drop rows with NAN in future coordinates xarray columns
geartypes_of_interest = ["TRAWLERS", 'SQUID_JIGGER']
df1 = df1[df1['geartype'].isin(geartypes_of_interest)] #We keep only the interesting gears for SWA intl

In [56]:
available_polygon = gpd.read_file("./data/available_SWA_fishing_area.geojson")

gdf1_points = gpd.GeoDataFrame(
    df1,
    geometry=gpd.points_from_xy(df1["lon"], df1["lat"]),  # lon = X, lat = Y
    crs="EPSG:4326"
)

In [57]:
gdf1_filtered = gpd.sjoin(
    gdf1_points,
    available_polygon,
    predicate="intersects",  
    how="inner"
)

In [58]:
gdf1_filtered

,date,lat,lon,hours,flag,geartype,geometry,index_right
5,2012-03-01,-50.3,-61.700001,22.676944,KOR,SQUID_JIGGER,POINT (-61.7 -50.3),0
17,2012-04-01,-50.4,-61.700001,16.267500,ESP,TRAWLERS,POINT (-61.7 -50.4),0
20,2012-04-01,-50.3,-61.000000,5.405278,ESP,TRAWLERS,POINT (-61 -50.3),0
26,2012-07-01,-50.9,-56.799999,2.992222,KOR,TRAWLERS,POINT (-56.8 -50.9),0
29,2012-05-01,-49.3,-60.700001,3.149444,KOR,SQUID_JIGGER,POINT (-60.7 -49.3),0
...,...,...,...,...,...,...,...,...
2854238,2024-10-01,-48.3,-60.700001,0.925556,ESP,TRAWLERS,POINT (-60.7 -48.3),0
2854240,2024-08-01,-46.2,-60.500000,2.372778,FLK,TRAWLERS,POINT (-60.5 -46.2),0
2854248,2024-01-01,-45.6,-60.400002,4.511389,KOR,SQUID_JIGGER,POINT (-60.4 -45.6),0
2854251,2024-10-01,-50.2,-58.000000,5.359167,FLK,TRAWLERS,POINT (-58 -50.2),0


In [59]:
lat_list = sorted(gdf1_filtered["lat"].unique().tolist())
print("lats:", lat_list)

lon_list = sorted(gdf1_filtered["lon"].unique().tolist())
print("lons:", lon_list)

lats: [-60.0, -59.9, -59.8, -59.7, -59.6, -59.5, -59.4, -59.3, -59.2, -59.1, -59.0, -58.9, -58.8, -58.7, -58.6, -58.5, -58.4, -58.3, -58.2, -58.1, -58.0, -57.9, -57.8, -57.7, -57.6, -57.5, -57.4, -57.3, -57.2, -57.1, -57.0, -56.9, -56.8, -56.7, -56.6, -56.5, -56.4, -56.3, -56.2, -56.0, -55.8, -55.7, -55.5, -55.4, -55.3, -55.2, -55.1, -55.0, -54.9, -54.8, -54.7, -54.6, -54.5, -54.4, -54.3, -54.2, -54.1, -54.0, -53.9, -53.8, -53.7, -53.6, -53.5, -53.4, -53.3, -53.2, -53.1, -53.0, -52.9, -52.8, -52.7, -52.6, -52.5, -52.4, -52.3, -52.2, -52.1, -52.0, -51.9, -51.8, -51.7, -51.6, -51.5, -51.4, -51.3, -51.2, -51.1, -51.0, -50.9, -50.8, -50.7, -50.6, -50.5, -50.4, -50.3, -50.2, -50.1, -50.0, -49.9, -49.8, -49.7, -49.6, -49.5, -49.4, -49.3, -49.2, -49.1, -49.0, -48.9, -48.8, -48.7, -48.6, -48.5, -48.4, -48.3, -48.2, -48.1, -48.0, -47.9, -47.8, -47.7, -47.6, -47.5, -47.4, -47.3, -47.2, -47.1, -47.0, -46.9, -46.8, -46.7, -46.6, -46.5, -46.4, -46.3, -46.2, -46.1, -46.0, -45.9, -45.8, -45.7, -45.6,

In [66]:
df = gdf1_filtered.drop(columns=["geometry", "index_right"]).copy()
df_agg = (
    df.groupby(["date", "lat", "lon", "flag", "geartype"], as_index=False)
      .agg(hours=('hours', 'sum'))
)

df_agg["lat"] = df_agg["lat"].round(1)
df_agg["lon"] = df_agg["lon"].round(1)

In [ ]:
dsxr = (
    df_agg
    .set_index(["date", "lat", "lon", "flag", "geartype"])
    .to_xarray()
    .rename({"date": "time"})
)

# Define chunk sizes for dask, time in 12-month chunks, lat and lon full size, geartype and flag one at a chunk
chunks = {"time": 12, "lat": -1, "lon": -1, "geartype":1, "flag":1}
dsxr = dsxr.chunk(chunks)

In [62]:
dsxr

<xarray.Dataset> Size: 2GB
Dimensions:   (time: 156, lat: 236, lon: 151, flag: 27, geartype: 2)
Coordinates:
  * time      (time) datetime64[ns] 1kB 2012-01-01 2012-02-01 ... 2024-12-01
  * lat       (lat) float64 2kB -60.0 -59.9 -59.8 -59.7 ... -36.3 -36.2 -35.9
  * lon       (lon) float64 1kB -65.0 -64.9 -64.8 -64.7 ... -50.2 -50.1 -50.0
  * flag      (flag) object 216B 'ALB' 'ARE' 'ARG' 'BLZ' ... 'UKR' 'URY' 'VUT'
  * geartype  (geartype) object 16B 'SQUID_JIGGER' 'TRAWLERS'
Data variables:
    hours     (time, lat, lon, flag, geartype) float64 2GB nan nan ... nan nan

In [71]:
lat_min_xr = dsxr.lat.min().item()
lat_max_xr = dsxr.lat.max().item()
lon_min_xr = dsxr.lon.min().item()
lon_max_xr = dsxr.lon.max().item()

bbox_xr = [lon_min_xr, lat_min_xr, lon_max_xr, lat_max_xr]
min_lon, min_lat, max_lon, max_lat = [-69.61, -60., -50., -32.45] #SW Atlantic bbox (roundeed to the second decimal)
bbox_SWA = [min_lon, min_lat, max_lon, max_lat]

print("Bounding box from xarray dataset:", bbox_xr)
print("SW Atlantic bounding box:", bbox_SWA) #Southwest Atlantic bounding box is greater, we need to regrid, we will mantain 0.1 degree resolution

lat_diff = np.diff(dsxr.lat)
lon_diff = np.diff(dsxr.lon)

print("Latitude spacing unique values:", np.unique(lat_diff))
print("Longitude spacing unique values:", np.unique(lon_diff))

Bounding box from xarray dataset: [-65.0, -60.0, -50.0, -35.9]
SW Atlantic bounding box: [-69.61, -60.0, -50.0, -32.45]
Latitude spacing unique values: [0.1 0.1 0.2 0.2 0.3]
Longitude spacing unique values: [0.1 0.1 0.1]


In [73]:
target_lons = np.arange(bbox_SWA[0], bbox_SWA[2] + 0.1, 0.1)
target_lats = np.arange(bbox_SWA[1], bbox_SWA[3] + 0.1, 0.1)

dsxr_expanded = dsxr.reindex(
    lat=target_lats,
    lon=target_lons
)

dsxr_expanded = dsxr_expanded.chunk({
    "time": 12, # 12 months at a chunk
    "lat": -1, # full size
    "lon": -1,
    "flag": 1, # one at a chunk
    "geartype": 1
}) #we need to call chunk again after reindexing (for saving the zarr)

In [ ]:
#Adding metadata and description to this zarr dataset
dsxr_expanded.attrs['title'] = "SW Atlantic GFW AIS Fishing Effort Data (2012-2024)"
dsxr_expanded.attrs["description"] = (
    """Monthly fishing hours per grid cell in the SW Atlantic, 
    aligned to 0.1° lat/lon grid
    only data from trawlers and squid jiggers vessels included,
    only data within the available fishing area is included which comprises
    international waters and claimed Falkland Islands EEZ."""
)

dsxr_expanded.attrs["source"] = "AIS-based fishing effort dataset"
dsxr_expanded.attrs["created_by"] = "Ruben Barriuso"
dsxr_expanded.attrs["resolution"] = "0.1x0.1 degree"
dsxr_expanded.attrs["date_created"] = "2025-12-29"
dsxr_expanded.attrs["source"] = "Global Fishing Watch (GFW)"

dsxr_expanded["hours"].attrs["units"] = "hours"
dsxr_expanded["hours"].attrs["long_name"] = "Fishing effort in AIS hours"
dsxr_expanded["hours"].attrs["description"] = (
    "Total monthly fishing effort observed in each grid cell from GFW"
)

In [ ]:

dsxr_expanded.to_zarr("./resources/GFW/GFW_AIS_Intl-FK_Trwl-Jgr_2012-2024_.zarr", 
                      consolidated=True)#creates a metadata file for faster access

c:\Users\rubar\miniconda3\envs\TFMenv\Lib\site-packages\zarr\api\asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
